In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_curve,
    auc,
    confusion_matrix,
)
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from matplotlib import rcParams
from tqdm import tqdm
import shap
import time

In [2]:
# ===================== DEVICE SETUP ===================== #
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [3]:
# ===================== PATHS AND CONFIGURATION ===================== #
model_name = "convnext"
OUTPUT_PATH = f"1_Feature_Extraction/{model_name}"
RESULTS_PATH = os.path.join("8_Batch_Size_Comparison")
os.makedirs(RESULTS_PATH, exist_ok=True)

# Load feature data
try:
    train_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(OUTPUT_PATH, f"test_features_{model_name}.csv"))
    print(f"Loaded {len(train_df)} training and {len(test_df)} testing samples")
except FileNotFoundError:
    print(f"ConvNext features not found, attempting to use available features")
    train_df = pd.read_csv(os.path.join(f"1_Feature_Extraction/{model_name}", f"train_features_{model_name}.csv"))
    test_df = pd.read_csv(os.path.join(f"1_Feature_Extraction/{model_name}", f"test_features_{model_name}.csv"))
    print(f"Using {model_name} features with {len(train_df)} training and {len(test_df)} testing samples")

Loaded 5712 training and 1311 testing samples


In [4]:
# Extract features and labels
feature_columns = [col for col in train_df.columns if col.startswith("feat_")]
X_train = train_df[feature_columns].values
y_train = train_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values
X_test = test_df[feature_columns].values
y_test = test_df["label"].map({"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}).values

CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
N_CLASSES = len(CLASSES)  # Should be 4
print(f"Number of classes: {N_CLASSES}")

Number of classes: 4


In [5]:
# Plot settings
rcParams["font.family"] = "Times New Roman"
rcParams["axes.titlesize"] = 28
rcParams["axes.titlepad"] = 20
rcParams["axes.labelsize"] = 23
rcParams["xtick.labelsize"] = 18
rcParams["ytick.labelsize"] = 18
rcParams["legend.fontsize"] = 16
rcParams["lines.linewidth"] = 3
rcParams["axes.linewidth"] = 2

In [6]:
# ===================== FEATURE SELECTION USING SHAP ===================== #
def apply_shap(X_train, X_test, n_features):
    print(f"Applying SHAP to select {n_features} features...")
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)
    feature_importance = rf_model.feature_importances_
    selected_indices = np.argsort(feature_importance)[::-1][:n_features]
    selected_indices = np.array(selected_indices, dtype=int)
    X_train_selected = X_train[:, selected_indices]
    X_test_selected = X_test[:, selected_indices]
    return X_train_selected, X_test_selected, selected_indices

In [7]:
# ===================== MODEL DEFINITION ===================== #
class FeatureDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [8]:
class Attention(nn.Module):
    def __init__(self, feature_dim):
        super(Attention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, feature_dim // 2),
            nn.Tanh(),
            nn.Linear(feature_dim // 2, 1),
            nn.Softmax(dim=1),
        )

    def forward(self, x):
        weights = self.attention(x)
        return (x * weights).sum(dim=1)

In [9]:
class AttGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=N_CLASSES):
        super(AttGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)  # Add sequence dimension
        x, _ = self.gru(x)
        x = self.attention(x)
        x = self.fc(x)
        return x

In [10]:
def train_model(model, train_loader, test_loader, batch_size, 
               learning_rate=0.001, num_epochs=100, weight_decay=1e-5):
    criterion = nn.CrossEntropyLoss()
    
    # Initialize AdamW optimizer with fixed learning rate
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    train_losses, test_losses, train_accs, test_accs = [], [], [], []
    
    # For measuring training time
    start_time = time.time()
    total_iterations = 0
    
    for epoch in tqdm(range(num_epochs), desc=f"Training AttGRU with batch size {batch_size}, AdamW LR={learning_rate}"):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        epoch_iterations = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            epoch_iterations += 1
            
        total_iterations += epoch_iterations
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        test_loss = test_loss / len(test_loader)
        test_acc = correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)

    training_time = time.time() - start_time
    iterations_per_epoch = total_iterations / num_epochs
    
    print(f"Training completed in {training_time:.2f} seconds")
    print(f"Average iterations per epoch: {iterations_per_epoch:.2f}")
    
    return train_losses, test_losses, train_accs, test_accs, training_time, iterations_per_epoch

In [11]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        outputs = model(X_test_tensor)
        _, y_pred = outputs.max(1)
        y_pred = y_pred.cpu().numpy()
        y_prob = torch.softmax(outputs, dim=1).cpu().numpy()

    metrics = {}
    metrics["ACC"] = accuracy_score(y_test, y_pred)
    metrics["AUC"] = roc_auc_score(y_test, y_prob, multi_class="ovr")
    metrics["PRE"] = precision_score(y_test, y_pred, average="macro")
    metrics["SN"] = recall_score(y_test, y_pred, average="macro")
    
    # Calculate specificity for multiclass
    cm = confusion_matrix(y_test, y_pred)
    specificity_scores = []
    for i in range(N_CLASSES):
        # True negatives are all elements of the confusion matrix except for the current class
        tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
        fp = np.sum(cm[:, i]) - cm[i, i]
        # Avoid division by zero
        if (tn + fp) == 0:
            specificity_scores.append(0)
        else:
            specificity_scores.append(tn / (tn + fp))
    metrics["SP"] = np.mean(specificity_scores)
    
    metrics["F1"] = f1_score(y_test, y_pred, average="macro")
    metrics["MCC"] = matthews_corrcoef(y_test, y_pred)
    return metrics, y_prob, y_pred

In [12]:
def plot_roc(y_test, y_prob, batch_size):
    y_test_bin = label_binarize(y_test, classes=range(N_CLASSES))
    fpr, tpr, roc_auc = {}, {}, {}
    plt.figure(figsize=(8, 8))

    for i in range(N_CLASSES):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        plt.plot(fpr[i], tpr[i], label=f"{CLASSES[i]} (AUC = {roc_auc[i]:.2f})")

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"AttGRU with Batch Size {batch_size} - ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.grid(True)
    
    # Save as PNG
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_BatchSize_{batch_size}_roc_curve.png"), dpi=1000, bbox_inches="tight")
    # Save as PDF
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_BatchSize_{batch_size}_roc_curve.pdf"), format='pdf', bbox_inches="tight")
    plt.close()

In [13]:
def plot_learning_curves(train_losses, test_losses, train_accs, test_accs, batch_size):
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(test_losses, label='Test Loss')
    plt.title(f'AdamW Batch Size {batch_size} - Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train Accuracy')
    plt.plot(test_accs, label='Test Accuracy')
    plt.title(f'AdamW Batch Size {batch_size} - Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save as PNG
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_BatchSize_{batch_size}_learning_curves.png"), dpi=1000, bbox_inches="tight")
    # Save as PDF
    plt.savefig(os.path.join(RESULTS_PATH, f"AttGRU_AdamW_BatchSize_{batch_size}_learning_curves.pdf"), format='pdf', bbox_inches="tight")
    plt.close()

In [14]:
print("Starting feature selection and batch size comparison with AdamW optimizer...")

# Define a fixed number of features to use
n_features = 800
print(f"\n===== Selecting {n_features} features using SHAP =====")
X_train_shap, X_test_shap, shap_indices = apply_shap(X_train, X_test, n_features=n_features)

Starting feature selection and batch size comparison with AdamW optimizer...

===== Selecting 800 features using SHAP =====
Applying SHAP to select 800 features...


In [15]:
# Define batch sizes to compare
batch_sizes = [16, 32, 64, 128]
batch_results = []

In [16]:
# Fixed parameters
learning_rate = 0.001
num_epochs = 100  # Fixed number of epochs for all batch sizes

# Prepare datasets
train_dataset = FeatureDataset(X_train_shap, y_train)
test_dataset = FeatureDataset(X_test_shap, y_test)

In [17]:
for batch_size in batch_sizes:
    print(f"\n===== Evaluating with batch size {batch_size} =====")
    
    # Initialize a new model for each batch size
    model = AttGRU(input_dim=X_train_shap.shape[1]).to(device)
    
    # Create data loaders with current batch size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Train with the current batch size using AdamW
    train_losses, test_losses, train_accs, test_accs, training_time, iterations_per_epoch = train_model(
        model, 
        train_loader, 
        test_loader,
        batch_size=batch_size,
        learning_rate=learning_rate,
        num_epochs=num_epochs
    )

    # Plot learning curves
    plot_learning_curves(train_losses, test_losses, train_accs, test_accs, batch_size)

    # Evaluate the model
    metrics, y_prob, y_pred = evaluate_model(model, X_test_shap, y_test)
    
    # Plot ROC curve
    plot_roc(y_test, y_prob, batch_size)
    
    # Calculate memory usage estimate (rough approximation)
    # This is a very simple approximation - actual memory usage depends on many factors
    estimated_memory_usage = batch_size * X_train_shap.shape[1] * 4 / (1024 * 1024)  # in MB
    
    # Save model
    torch.save(model.state_dict(), os.path.join(RESULTS_PATH, f"AttGRU_AdamW_BatchSize_{batch_size}_model.pth"))

    # Save results
    batch_results.append(
        {
            "Batch_Size": batch_size,
            "Training_Time": training_time,
            "Iterations_Per_Epoch": iterations_per_epoch,
            "Est_Memory_MB": estimated_memory_usage,
            "ACC": metrics["ACC"],
            "AUC": metrics["AUC"],
            "PRE": metrics["PRE"],
            "SN": metrics["SN"],
            "SP": metrics["SP"],
            "F1": metrics["F1"],
            "MCC": metrics["MCC"],
        }
    )
    print(f"AttGRU with batch size {batch_size} - Performance Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")
    print(f"Iterations Per Epoch: {iterations_per_epoch:.2f}")
    print(f"Estimated Memory Usage: {estimated_memory_usage:.2f} MB")


===== Evaluating with batch size 16 =====


Training AttGRU with batch size 16, AdamW LR=0.001: 100%|██████████| 100/100 [04:22<00:00,  2.62s/it]


Training completed in 262.51 seconds
Average iterations per epoch: 357.00
AttGRU with batch size 16 - Performance Metrics:
ACC: 0.9130
AUC: 0.9875
PRE: 0.9093
SN: 0.9062
SP: 0.9713
F1: 0.9073
MCC: 0.8835
Training Time: 262.51 seconds
Iterations Per Epoch: 357.00
Estimated Memory Usage: 0.05 MB

===== Evaluating with batch size 32 =====


Training AttGRU with batch size 32, AdamW LR=0.001: 100%|██████████| 100/100 [02:55<00:00,  1.75s/it]


Training completed in 175.28 seconds
Average iterations per epoch: 179.00
AttGRU with batch size 32 - Performance Metrics:
ACC: 0.9016
AUC: 0.9859
PRE: 0.8966
SN: 0.8938
SP: 0.9674
F1: 0.8948
MCC: 0.8681
Training Time: 175.28 seconds
Iterations Per Epoch: 179.00
Estimated Memory Usage: 0.10 MB

===== Evaluating with batch size 64 =====


Training AttGRU with batch size 64, AdamW LR=0.001: 100%|██████████| 100/100 [01:15<00:00,  1.33it/s]


Training completed in 75.04 seconds
Average iterations per epoch: 90.00
AttGRU with batch size 64 - Performance Metrics:
ACC: 0.9062
AUC: 0.9857
PRE: 0.9029
SN: 0.8992
SP: 0.9690
F1: 0.9003
MCC: 0.8744
Training Time: 75.04 seconds
Iterations Per Epoch: 90.00
Estimated Memory Usage: 0.20 MB

===== Evaluating with batch size 128 =====


Training AttGRU with batch size 128, AdamW LR=0.001: 100%|██████████| 100/100 [00:40<00:00,  2.46it/s]


Training completed in 40.57 seconds
Average iterations per epoch: 45.00
AttGRU with batch size 128 - Performance Metrics:
ACC: 0.9039
AUC: 0.9850
PRE: 0.9006
SN: 0.8963
SP: 0.9680
F1: 0.8979
MCC: 0.8712
Training Time: 40.57 seconds
Iterations Per Epoch: 45.00
Estimated Memory Usage: 0.39 MB


In [18]:
# Save comparison results to CSV
batch_df = pd.DataFrame(batch_results)
batch_df.to_csv(os.path.join(RESULTS_PATH, "AttGRU_AdamW_batch_size_comparison.csv"), index=False)
print(f"Saved batch size comparison results to {os.path.join(RESULTS_PATH, 'AttGRU_AdamW_batch_size_comparison.csv')}")

Saved batch size comparison results to 8_Batch_Size_Comparison/AttGRU_AdamW_batch_size_comparison.csv


In [19]:
# Plot comparison results for performance metrics
plt.figure(figsize=(14, 8))
metrics_to_plot = ["ACC", "AUC", "SP", "SN", "F1", "MCC"]
x = np.arange(len(batch_sizes))
width = 0.15

for i, metric in enumerate(metrics_to_plot):
    values = [result[metric] for result in batch_results]
    plt.bar(x + i * width, values, width, label=metric)

plt.xlabel("Batch Size")
plt.ylabel("Score")
plt.title("AttGRU Performance with AdamW at Different Batch Sizes")
plt.xticks(x + width * (len(metrics_to_plot) - 1) / 2, [str(bs) for bs in batch_sizes])
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_batch_size_comparison.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_batch_size_comparison.pdf"), dpi=1000, format='pdf', bbox_inches="tight")
plt.close()

In [20]:
# Plot training time comparison
plt.figure(figsize=(14, 6))

# Create subplots
plt.subplot(1, 2, 1)
plt.plot(batch_sizes, [result["Training_Time"] for result in batch_results], 'o-', linewidth=2, markersize=10)
plt.grid(True)
plt.xlabel("Batch Size")
plt.ylabel("Training Time (seconds)")
plt.title("Training Time vs Batch Size")

plt.subplot(1, 2, 2)
plt.plot(batch_sizes, [result["Iterations_Per_Epoch"] for result in batch_results], 'o-', linewidth=2, markersize=10)
plt.grid(True)
plt.xlabel("Batch Size")
plt.ylabel("Iterations Per Epoch")
plt.title("Iterations Per Epoch vs Batch Size")

plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_batch_training_efficiency.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_batch_training_efficiency.pdf"), dpi=1000, format='pdf', bbox_inches="tight")
plt.close()

In [21]:
# Plot metrics vs batch size with training time as point size
plt.figure(figsize=(14, 12))
training_times = np.array([result["Training_Time"] for result in batch_results])
normalized_times = 50 + 250 * (training_times / training_times.max())

for i, metric in enumerate(metrics_to_plot):
    plt.subplot(2, 3, i+1)
    metric_values = [result[metric] for result in batch_results]
    
    # Scatter plot with size representing training time
    plt.scatter(batch_sizes, metric_values, s=normalized_times, alpha=0.7)
    plt.plot(batch_sizes, metric_values, 'o-', linewidth=1, alpha=0.5)
    
    # Add batch size annotations
    for j, bs in enumerate(batch_sizes):
        plt.annotate(f"{bs}", 
                     (batch_sizes[j], metric_values[j]),
                     textcoords="offset points",
                     xytext=(0,7), 
                     ha='center',
                     fontsize=9)
    
    plt.grid(True)
    plt.xlabel("Batch Size")
    plt.ylabel(metric)
    plt.title(f"{metric} vs Batch Size")

plt.tight_layout()

# Save as PNG
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_metrics_vs_batch_size.png"), dpi=1000, bbox_inches="tight")
# Save as PDF
plt.savefig(os.path.join(RESULTS_PATH, "AttGRU_AdamW_metrics_vs_batch_size.pdf"), dpi=1000, format='pdf', bbox_inches="tight")
plt.close()